In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
!pip install langchain

In [ ]:
import sys
print(sys.executable)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  # correct current path
print("SUCCESS")

In [ ]:
# Extract text from pdf files
def load_pdf_files(data):
    loader = DirectoryLoader(data,glob="*.pdf",loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [ ]:
import os
print(os.getcwd())

In [ ]:
print(os.listdir())

In [ ]:
os.chdir("../")

In [ ]:
print(os.listdir())

In [ ]:
import os
project_root = r"C:\VIZUARA-ML-AI\projects\medical_chatBot"
print(os.listdir(project_root))

In [ ]:
import os
project_root = r"C:\VIZUARA-ML-AI\projects\medical_chatBot"
extracted_data = load_pdf_files(os.path.join(project_root, "data"))
print(f"Loaded {len(extracted_data)} pages")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of document objects, return a new list of document objects containing only 'source' in metadata and the original 'page_content
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20
    )
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

In [ ]:
minimal_docs

In [ ]:
text_chunks = text_split(minimal_docs)
print(f"Number of chunks:{len(text_chunks)}")

In [ ]:
text_chunks

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
def download_embeddings():
    """
    Download and return huggingface embedding model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings
embedding = download_embeddings()

In [ ]:
embedding

In [ ]:
vector = embedding.embed_query("Hello World")

In [ ]:
vector

In [ ]:
print("Vector Length:", len(vector))

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embedding,
    index_name=index_name
)

In [ ]:
# Load Existing Index
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone Index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [ ]:
# Add more data to the existing Pinecone Index
tariq = Document(
    page_content = "Tariq is an AI Engineer with expertise in machine learning and natural language processing.",
    metadata = {"source": "tariq.txt"}
)

In [ ]:
docsearch.add_documents(documents=[tariq])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

In [ ]:
from langchain_openai import ChatOpenAI

chatmodel = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are a Medical Assistant for question answering tasks."
    "Use the following pieces of retrieved context to answer "
    "the question.If you dont know the answer, say that you "
    "don't know.Use three sentences maximum to answer the question."
    "answer concise "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [ ]:
question_answer_chain = create_stuff_documents_chain(chatmodel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

key = os.environ.get("OPENAI_API_KEY")
print(repr(key))       # repr() reveals hidden quotes/whitespace that print() hides
print(len(key) if key else "No key found")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "say hi"}]
    )
    print("✅ Key is working:", response.choices[0].message.content)
except Exception as e:
    print("❌ Key failed:", e)

In [ ]:
response = rag_chain.invoke({"input":" what is Acromegaly and gigantism?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input":" what is Acne?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input":" who is TARIQ HUSAIN KHAN?"})
print(response["answer"])